# PRISMA Walkthrough: From Natural Language to SPARQL

This notebook traces a single natural language query through all three stages of the PRISMA pipeline, showing every intermediate artifact produced along the way.

---

## What is PRISMA?

**PRISMA** (*Pattern-based Rules for Intent-driven SPARQL with Multiple-resource Assembly*) is a hybrid NL2SPARQL system designed for the **LiITA Knowledge Base** — a linked lexical resource for Italian. It connects six resources:

| Resource | Type | Contents |
|---|---|---|
| **LiITA Lemma Bank** | Local | 80K+ Italian lemmas with POS, gender, inflection |
| **CompL-it** | External (SERVICE) | 101K+ entries with definitions and 137 semantic relation types |
| **Sentix** | Local | 63K+ sentiment / polarity entries |
| **ELIta** | Local | Emotion classification (Joy, Sadness, Anger, …) |
| **Parmigiano** | Local | Italian ↔ Parmigiano dialect translations |
| **Sicilian** | Local | Italian ↔ Sicilian dialect translations |

## The Three-Stage Pipeline

```
 Natural Language Query
         │
         ▼
 ┌─────────────────────┐
 │  Stage 1            │  LLM is used ONLY here.
 │  Intent Analysis    │  The query is classified and its
 │  (stochastic)       │  parameters are extracted into JSON.
 └──────────┬──────────┘
            │  intent dict (JSON)
            ▼
 ┌─────────────────────┐
 │  Stage 2            │  Pure deterministic rules.
 │  Pattern            │  The intent is mapped to an ordered
 │  Orchestration      │  list of pattern steps to execute.
 └──────────┬──────────┘
            │  ExecutionPlan
            ▼
 ┌─────────────────────┐
 │  Stage 3            │  Each step calls a pattern tool,
 │  SPARQL             │  which generates a SPARQL fragment.
 │  Assembly           │  Fragments are composed into the
 └─────────────────────┘  final complete SPARQL query.
            │
            ▼
       SPARQL Query
```

**Key insight:** only Stage 1 is stochastic (LLM-driven). Stages 2 and 3 are fully deterministic — the same intent always produces the same SPARQL.

---

### Example query we will trace

> *"Find the hyponyms of 'colore'"*

This is a `complit_semantic` query. It requires LiITA (to anchor the lemma) and CompL-it (accessed via a SPARQL `SERVICE` clause) to navigate semantic relations, making it a good representative of a mid-complexity multi-resource query.

---
## 0 — Setup

In [1]:
import sys, json, textwrap, time
from pathlib import Path

# Make sure the project root is on sys.path
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root : {ROOT}")
print(f"Python path  : {sys.path[0]}")

Project root : C:\gio\Tesi\prisma-per-liita
Python path  : C:\gio\Tesi\prisma-per-liita


In [2]:
# ============================================================
# Configuration
# ============================================================

# Set USE_MOCK_LLM = False to call a real LLM.
# The mock returns a pre-computed response so the notebook
# is fully reproducible without any API key.
USE_MOCK_LLM = True

# Real LLM settings (ignored when USE_MOCK_LLM = True)
PROVIDER = "mistral"   # "mistral" | "anthropic" | "openai" | "gemini" | "ollama"
API_KEY  = "your_api_key" # or set directly; falls back to env var
MODEL    = None          # None = provider default

EXAMPLE_QUERY = "Find the hyponyms of 'colore'"

print(f"Query        : {EXAMPLE_QUERY!r}")
print(f"LLM mode     : {'mock (offline)' if USE_MOCK_LLM else PROVIDER}")

Query        : "Find the hyponyms of 'colore'"
LLM mode     : mock (offline)


In [3]:
from shared import create_llm_client

# Pre-computed response for mock mode
# This is the JSON that a real LLM would return for EXAMPLE_QUERY.
MOCK_JSON_RESPONSE = json.dumps({
    "query_type": "complit_semantic",
    "required_resources": ["complit", "liita_lemma_bank"],
    "lemma": "colore",
    "lemma_b": None,
    "lemmas": None,
    "semantic_relation": "hyponym",
    "filters": [],
    "aggregation": None,
    "retrieve_definitions": True,
    "retrieve_examples": False,
    "include_italian_written_rep": True,
    "complexity_score": 3,
    "user_query": "Find the hyponyms of 'colore'",
    "reasoning": (
        "Semantic relation query: hyponyms of 'colore' retrieved from CompL-it "
        "(via SERVICE clause), then bridged to LiITA lemmas."
    )
}, indent=2)


class MockLLM:
    """Minimal mock LLM that returns a fixed pre-computed response."""
    provider_name = "Mock"
    model         = "mock-1.0"

    def complete(self, prompt, system=None, temperature=None, max_tokens=None, **kwargs):
        return MOCK_JSON_RESPONSE


if USE_MOCK_LLM:
    llm = MockLLM()
    print(f"LLM : {llm.provider_name} / {llm.model}  (offline — no API key needed)")
else:
    llm = create_llm_client(provider=PROVIDER, api_key=API_KEY, model=MODEL)
    print(f"LLM : {llm.provider_name} / {llm.model}")

LLM : Mock / mock-1.0  (offline — no API key needed)


---
## Stage 1 — Intent Analysis (LLM)

The `IntentAnalyzer` is the **only component** in PRISMA that calls a language model. Its job is deliberately narrow: it does **not** generate SPARQL. It only classifies the query and extracts structured parameters into a JSON object.

The LLM call is structured as a standard two-part prompt:
- **System message** — describes the task, the available resources, the output schema, and constraints.
- **User message** — a set of few-shot examples followed by the actual query to classify.

### 1.1 — The System Prompt

The system prompt defines everything the LLM needs to know to produce a valid intent JSON. It includes:
- Description of each available resource (LiITA, CompL-it, Sentix, ELIta, dialects)
- The list of valid `query_type` values
- The `filters` array specification
- The exact JSON output schema
- Critical rules (no markdown, exact enum values, aggregation only when explicit, …)

In [4]:
from prisma.intent_analyzer import INTENT_ANALYZER_SYSTEM_PROMPT

print(f"System prompt length : {len(INTENT_ANALYZER_SYSTEM_PROMPT):,} characters\n")
print("=" * 70)
print("SYSTEM PROMPT (full)")
print("=" * 70)
print(INTENT_ANALYZER_SYSTEM_PROMPT)

System prompt length : 9,975 characters

SYSTEM PROMPT (full)
You are an expert intent analyzer for the LiITA Knowledge Base, a linguistic resource for Italian that links multiple lexical resources.

Your ONLY task is to analyze natural language queries about Italian linguistics and produce structured JSON output. You do NOT generate SPARQL queries.

# AVAILABLE RESOURCES

1. **LiITA Lemma Bank** (Local)
   - Italian lemmas with part-of-speech, gender, inflection type tags
   - Written representations
   - Basic filtering capabilities

2. **CompL-it** (External - requires SERVICE clause)
   - Computational lexicon with 101K+ entries
   - Natural language definitions
   - Semantic relations: hypernyms, hyponyms, meronyms, holonyms, synonyms
   - 137 types of semantic relations
   - Usage examples

3. **Parmigiano Lexicon** (Local)
   - Italian ↔ Parmigiano dialect translations
   - Pattern-based searching available
   - Linked via LiITA lemmas

4. **Sicilian Lexicon** (Local)
   - Itali

### 1.2 — The User Message (few-shot examples + the query)

The user message is built by `IntentAnalyzer._build_user_prompt()`. It prepends up to 12 few-shot (query → intent JSON) examples before the actual query. This is the **complete text** sent as the user turn to the LLM.

Below we show a truncated version (first example only) to keep the output readable.

In [5]:
from prisma.intent_analyzer import IntentAnalyzer, INTENT_EXAMPLES

analyzer = IntentAnalyzer(llm)

# Build the prompt exactly as the analyzer does internally
# (include_examples=5 is the default in Translator.translate)
NUM_EXAMPLES = 5
user_prompt = analyzer._build_user_prompt(EXAMPLE_QUERY, num_examples=NUM_EXAMPLES)

print(f"Number of few-shot examples included : {NUM_EXAMPLES}")
print(f"Total user message length            : {len(user_prompt):,} characters\n")
print("=" * 70)
print("USER MESSAGE (first example + the query, rest truncated)")
print("=" * 70)

# Find the boundary between the examples block and the final query block
split_marker = "# QUERY TO ANALYZE"
idx = user_prompt.find(split_marker)

# Show first example only, then skip to the query block
first_example_end = user_prompt.find("## Example 2")
if first_example_end == -1:
    first_example_end = idx

print(user_prompt[:first_example_end])
print("\n... [ examples 2 – {} omitted for brevity ] ...\n".format(NUM_EXAMPLES))
print(user_prompt[idx:])

Number of few-shot examples included : 5
Total user message length            : 3,894 characters

USER MESSAGE (first example + the query, rest truncated)
# CLASSIFICATION EXAMPLES

## Example 1
Query: "Find all masculine nouns ending with 'a'"
Intent JSON:
```json
{
  "query_type": "basic_lemma_lookup",
  "required_resources": [
    "liita_lemma_bank"
  ],
  "lemma": null,
  "semantic_relation": null,
  "filters": [
    {
      "property": "pos",
      "value": "noun"
    },
    {
      "property": "gender",
      "value": "masculine"
    },
    {
      "pattern": "a$",
      "pattern_type": "ends_with"
    }
  ],
  "aggregation": null,
  "retrieve_definitions": false,
  "retrieve_examples": false,
  "include_italian_written_rep": true,
  "complexity_score": 2,
  "user_query": "Find all masculine nouns ending with 'a'",
  "reasoning": "LiITA lemma search with POS, gender, and written form pattern filters"
}
```



... [ examples 2 – 5 omitted for brevity ] ...

# QUERY TO ANALYZE

Que

### 1.3 — Calling the LLM

The analyzer calls `llm.complete(prompt=user_prompt, system=INTENT_ANALYZER_SYSTEM_PROMPT, temperature=0.0)`. Temperature is fixed at 0 to make classification deterministic.

In [6]:
print(f"Sending request to {llm.provider_name} ({llm.model}) ...")
print(f"Query : {EXAMPLE_QUERY!r}\n")

t0 = time.time()
raw_llm_response = llm.complete(
    prompt=user_prompt,
    system=INTENT_ANALYZER_SYSTEM_PROMPT,
    temperature=0.0
)
elapsed_ms = (time.time() - t0) * 1000

print(f"Response received in {elapsed_ms:.0f} ms\n")
print("=" * 70)
print("RAW LLM RESPONSE")
print("=" * 70)
print(raw_llm_response)

Sending request to Mock (mock-1.0) ...
Query : "Find the hyponyms of 'colore'"

Response received in 0 ms

RAW LLM RESPONSE
{
  "query_type": "complit_semantic",
  "required_resources": [
    "complit",
    "liita_lemma_bank"
  ],
  "lemma": "colore",
  "lemma_b": null,
  "lemmas": null,
  "semantic_relation": "hyponym",
  "filters": [],
  "aggregation": null,
  "retrieve_definitions": true,
  "retrieve_examples": false,
  "include_italian_written_rep": true,
  "complexity_score": 3,
  "user_query": "Find the hyponyms of 'colore'",
  "reasoning": "Semantic relation query: hyponyms of 'colore' retrieved from CompL-it (via SERVICE clause), then bridged to LiITA lemmas."
}


### 1.4 — Parsing the Response into an Intent Dict

The raw response is passed through `_parse_response()` which strips any accidental markdown fences and calls `json.loads()`. The result is then validated by `_validate_intent()` which checks required fields and enum values.

In [7]:
# Parse the raw JSON string
intent_dict, parse_warnings = analyzer._parse_response(raw_llm_response)

# Validate required fields and enum values
validation_warnings = analyzer._validate_intent(intent_dict)

# Ensure user_query is stamped on the dict
if not intent_dict.get("user_query"):
    intent_dict["user_query"] = EXAMPLE_QUERY

all_warnings = parse_warnings + validation_warnings

print("=" * 70)
print("PARSED INTENT DICT")
print("=" * 70)
print(json.dumps(intent_dict, indent=2, ensure_ascii=False))

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)
if all_warnings:
    print(f"  {len(all_warnings)} warning(s):")
    for w in all_warnings:
        print(f"    - {w}")
else:
    print("  No warnings — intent is valid.")

PARSED INTENT DICT
{
  "query_type": "complit_semantic",
  "required_resources": [
    "complit",
    "liita_lemma_bank"
  ],
  "lemma": "colore",
  "lemma_b": null,
  "lemmas": null,
  "semantic_relation": "hyponym",
  "filters": [],
  "aggregation": null,
  "retrieve_definitions": true,
  "retrieve_examples": false,
  "include_italian_written_rep": true,
  "complexity_score": 3,
  "user_query": "Find the hyponyms of 'colore'",
  "reasoning": "Semantic relation query: hyponyms of 'colore' retrieved from CompL-it (via SERVICE clause), then bridged to LiITA lemmas."
}

VALIDATION
  No warnings — intent is valid.


### 1.5 — Converting the Dict to a Typed Intent Object

`IntentConverter.dict_to_intent()` converts the raw dict into a strongly-typed `Intent` dataclass. It also validates and transforms the `filters` array using `FilterValidator`, and parses all enum fields (`QueryType`, `ResourceType`, `SemanticRelationType`).

In [8]:
from prisma.translator import IntentConverter

converter = IntentConverter()
intent, filter_warnings = converter.dict_to_intent(intent_dict)

print("=" * 70)
print("INTENT OBJECT")
print("=" * 70)
print(f"  query_type           : {intent.query_type}")
print(f"  required_resources   : {[r.value for r in intent.required_resources]}")
print(f"  lemma                : {intent.lemma!r}")
print(f"  lemma_b              : {intent.lemma_b!r}")
print(f"  semantic_relation    : {intent.semantic_relation}")
print(f"  pos                  : {intent.pos!r}")
print(f"  gender               : {intent.gender!r}")
print(f"  filters              : {intent.filters}")
print(f"  aggregation          : {intent.aggregation}")
print(f"  retrieve_definitions : {intent.retrieve_definitions}")
print(f"  include_italian_wr   : {intent.include_italian_written_rep}")
print(f"  complexity_score     : {intent.complexity_score}")
print(f"  user_query           : {intent.user_query!r}")

if filter_warnings:
    print(f"\n  Filter warnings: {filter_warnings}")

INTENT OBJECT
  query_type           : QueryType.COMPLIT_SEMANTIC
  required_resources   : ['complit', 'liita_lemma_bank']
  lemma                : 'colore'
  lemma_b              : None
  semantic_relation    : SemanticRelationType.HYPONYM
  pos                  : None
  gender               : None
  filters              : []
  aggregation          : None
  retrieve_definitions : True
  include_italian_wr   : True
  complexity_score     : 3
  user_query           : "Find the hyponyms of 'colore'"


---
## Stage 2 — Pattern Orchestration (Rules)

The `PatternOrchestrator` is a **pure rule engine** — no LLM, no randomness. It takes the `Intent` object and applies deterministic rules to select which pattern tools are needed and in what order.

The output is an `ExecutionPlan`: an ordered list of `PatternStep` objects, each specifying:
- `tool_name` — which pattern tool to call
- `parameters` — what to pass to it
- `description` — a human-readable explanation
- `depends_on` — step dependencies (for understanding data flow)

### 2.1 — Which Rule Fires?

The orchestrator routes by `query_type`. For `complit_semantic` it calls `_plan_complit_semantic()`, which applies these rules:

1. **Base component**: a semantic relation exists → use `complit_semantic_relation` (SERVICE)
2. **Bridge**: CompL-it + any other resource → always add `bridge_complit_to_liita`
3. **Italian written rep**: CompL-it was used → add `italian_written_rep` to retrieve the surface form
4. **Enrichments**: walk `required_resources`; add `sentix_linking`, `elita_linking`, dialect tools as needed
5. **Aggregation**: if `intent.aggregation` is set, append the `aggregation` step

In [9]:
from prisma.orchestrator import PatternOrchestrator

orchestrator = PatternOrchestrator()
plan = orchestrator.create_plan(intent)

print("=" * 70)
print(f"EXECUTION PLAN  ({len(plan.steps)} step(s))")
print("=" * 70)

for step in plan.steps:
    print(f"\nStep {step.step_number}: {step.tool_name}")
    print(f"  description  : {step.description}")
    if step.depends_on:
        print(f"  depends on   : step(s) {step.depends_on}")
    print(f"  parameters   :")
    for k, v in step.parameters.items():
        print(f"    {k:22s} = {v!r}")

EXECUTION PLAN  (3 step(s))

Step 1: complit_semantic_relation
  description  : Find hyponyms of 'colore' in CompL-it
  parameters   :
    lemma                  = 'colore'
    relation_type          = 'hyponym'
    pos                    = 'noun'
    retrieve_definitions   = True

Step 2: bridge_complit_to_liita
  description  : Bridge related words to LiITA lemmas
  depends on   : step(s) [1]
  parameters   :
    source_var             = '?relatedWord'

Step 3: italian_written_rep
  description  : Get Italian written representations
  depends on   : step(s) [2]
  parameters   :
    lemma_var              = '?liitaLemma'
    output_var             = '?italianWR'


### 2.2 — Understanding the Plan

Let us print a compact summary of the data flow between steps.

In [10]:
print("Data-flow summary\n")
print("  [NL query]")
print("      │  (LLM analysis — Stage 1)")
print(f"      ▼")

for i, step in enumerate(plan.steps):
    connector = "  " + ("└" if i == len(plan.steps) - 1 else "├") + "─"
    print(f"{connector} Step {step.step_number}: {step.tool_name}")
    print(f"  {'│' if i < len(plan.steps) - 1 else ' '}     └─ {step.description}")

print("\n  [SPARQL query]  ←  composed by assembler (Stage 3)")

Data-flow summary

  [NL query]
      │  (LLM analysis — Stage 1)
      ▼
  ├─ Step 1: complit_semantic_relation
  │     └─ Find hyponyms of 'colore' in CompL-it
  ├─ Step 2: bridge_complit_to_liita
  │     └─ Bridge related words to LiITA lemmas
  └─ Step 3: italian_written_rep
        └─ Get Italian written representations

  [SPARQL query]  ←  composed by assembler (Stage 3)


---
## Stage 3 — SPARQL Assembly

The `PatternAssembler` iterates over the `ExecutionPlan` steps. For each step it:

1. Looks up the pattern tool in the `PatternToolRegistry`
2. Calls `tool.generate(**step.parameters)` → returns a `PatternFragment`
3. Validates that all required input variables are available from previous fragments
4. Accumulates the fragment

After all fragments are collected, it:
- Composes the WHERE clause (by concatenating fragment SPARQL)
- Builds the SELECT clause from output variables
- Adds PREFIX declarations
- Appends GROUP BY / ORDER BY / LIMIT if an aggregation fragment exists

### 3.1 — The Pattern Tool Registry

All pattern tools are registered at startup. Each tool is a self-contained SPARQL template.

In [11]:
from prisma.pattern_tools import PatternToolRegistry
from prisma.assembler import PatternAssembler

registry = PatternToolRegistry()
tool_names = registry.list_tools()

print(f"Registered pattern tools ({len(tool_names)} total):\n")
for name in tool_names:
    tool = registry.get(name)
    print(f"  {name:45s}  {tool.description[:60]}")

Registered pattern tools (15 total):

  complit_definition_search                      Search CompL-it entries by definition pattern
  complit_semantic_relation                      Navigate semantic relations (hypernymy, meronymy, etc.) in C
  complit_word_sense_lookup                      Look up a word in CompL-it and retrieve all its senses and d
  complit_definition_linking                     Link LiITA lemmas to CompL-it definitions
  complit_relation_between_lemmas                Check if a semantic relation exists between two lemmas in Co
  bridge_complit_to_liita                        Bridge CompL-it word entries to LiITA lemmas
  parmigiano_translation                         Link Italian lemmas to Parmigiano dialect translations
  parmigiano_pattern_search                      Search Parmigiano lemmas by pattern, optionally link to Ital
  sicilian_translation                           Link Italian lemmas to Sicilian dialect translations
  sicilian_pattern_search           

### 3.2 — Generating Fragments Step by Step

We now manually walk through each `PatternStep`, call the corresponding tool, and inspect the `PatternFragment` it produces — the exact same process the `PatternAssembler` follows internally.

In [12]:
print("=" * 70)
print("PATTERN FRAGMENTS (one per execution step)")
print("=" * 70)

manual_fragments = []

for step in plan.steps:
    tool = registry.get(step.tool_name)
    if tool is None:
        print(f"\n  Step {step.step_number}: WARNING — tool '{step.tool_name}' not found!")
        continue

    # Generate the fragment
    fragment = tool.generate(**step.parameters)
    manual_fragments.append(fragment)

    separator = "─" * 70
    print(f"\n{separator}")
    print(f"  Step {step.step_number}  →  Tool: {tool.name}")
    print(f"  Description      : {step.description}")
    print(f"  Tool description : {tool.description}")
    print(f"{separator}")

    print(f"  Pattern fragment: {fragment.pattern_name}")
    print(f"    needs_service_clause : {fragment.needs_service_clause}")
    print(f"    required_prefixes    : {sorted(fragment.required_prefixes)}")
    print(f"    input_vars           : {[v.name + ' (' + v.type.value + ')' for v in fragment.input_vars]}")
    print(f"    output_vars          : {[v.name + ' (' + v.type.value + ')' for v in fragment.output_vars]}")

    print(f"\n  SPARQL fragment:")
    for line in fragment.sparql.splitlines():
        print(f"    {line}")

PATTERN FRAGMENTS (one per execution step)

──────────────────────────────────────────────────────────────────────
  Step 1  →  Tool: complit_semantic_relation
  Description      : Find hyponyms of 'colore' in CompL-it
  Tool description : Navigate semantic relations (hypernymy, meronymy, etc.) in CompL-it
──────────────────────────────────────────────────────────────────────
  Pattern fragment: complit_semantic_relation
    needs_service_clause : True
    required_prefixes    : ['lexinfo', 'ontolex', 'rdfs', 'skos']
    input_vars           : []
    output_vars          : ['?relatedWord (word)', '?relatedSense (sense)', '?definition (definition)']

  SPARQL fragment:
    
      SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
        ?word a ontolex:Word ;
              lexinfo:partOfSpeech [ rdfs:label ?pos ] ;
              ontolex:sense ?sense ;
              ontolex:canonicalForm [ ontolex:writtenRep ?lemma ] .
    
        ?sense lexinfo:hypernym ?relatedSense .
        OPTI

### 3.3 — Assembling the Final SPARQL Query

Now we call `assembler.assemble(plan)` which performs the full composition: prefixes + SELECT + WHERE + optional modifiers.

In [13]:
assembler = PatternAssembler(registry)
assembly_result = assembler.assemble(plan)

print("=" * 70)
print("ASSEMBLY METADATA")
print("=" * 70)
print(f"  Fragments generated : {assembly_result.metadata['num_fragments']}")
print(f"  Uses SERVICE clause : {assembly_result.metadata['uses_service']}")
print(f"  Has aggregation     : {assembly_result.metadata['has_aggregation']}")
if assembly_result.warnings:
    print(f"  Warnings            : {assembly_result.warnings}")
else:
    print(f"  Warnings            : none")

print("\n" + "=" * 70)
print("FINAL SPARQL QUERY")
print("=" * 70)
print(assembly_result.sparql_query)

ASSEMBLY METADATA
  Fragments generated : 3
  Uses SERVICE clause : True
  Has aggregation     : False
  Warnings            : none

FINAL SPARQL QUERY
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?relatedWord ?relatedSense ?definition ?liitaLemma ?italianWR
WHERE {
SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
    ?word a ontolex:Word ;
          lexinfo:partOfSpeech [ rdfs:label ?pos ] ;
          ontolex:sense ?sense ;
          ontolex:canonicalForm [ ontolex:writtenRep ?lemma ] .

    ?sense lexinfo:hypernym ?relatedSense .
    OPTIONAL {
      ?relatedSense skos:definition ?definition
    } .

    FILTER(str(?pos) = "noun") .
    FILTER(str(?lemma) = "colore") .

    ?relatedWord ontolex:sense ?relatedSense .
  }
?relatedWord ontolex:canonicalForm ?liitaLemma .
?liitaLemma ontolex:writtenRe

### 3.4 — How the WHERE Clause Was Composed

The assembler simply concatenates the non-aggregation fragment SPARQL strings inside a `WHERE { … }` block. Here we show the composition step explicitly.

In [14]:
non_agg_fragments = [f for f in assembly_result.fragments if not f.metadata.get('is_aggregation')]

print("WHERE clause composition\n")
print("WHERE {")
for i, frag in enumerate(non_agg_fragments):
    label = f"  # ── Fragment {i+1}: {frag.pattern_name}"
    print(label)
    for line in frag.sparql.strip().splitlines():
        print(f"  {line}")
    print()
print("}")

WHERE clause composition

WHERE {
  # ── Fragment 1: complit_semantic_relation
  SERVICE <https://klab.ilc.cnr.it/graphdb-compl-it/> {
      ?word a ontolex:Word ;
            lexinfo:partOfSpeech [ rdfs:label ?pos ] ;
            ontolex:sense ?sense ;
            ontolex:canonicalForm [ ontolex:writtenRep ?lemma ] .
  
      ?sense lexinfo:hypernym ?relatedSense .
      OPTIONAL {
        ?relatedSense skos:definition ?definition
      } .
  
      FILTER(str(?pos) = "noun") .
      FILTER(str(?lemma) = "colore") .
  
      ?relatedWord ontolex:sense ?relatedSense .
    }

  # ── Fragment 2: bridge_complit_to_liita
  ?relatedWord ontolex:canonicalForm ?liitaLemma .

  # ── Fragment 3: italian_written_rep
  ?liitaLemma ontolex:writtenRep ?italianWR .

}


---
## 4 — High-Level Comparison

The `Translator` class is just a convenience wrapper that chains Stages 1, 2, and 3. Running it should produce an identical result to our step-by-step walkthrough above.

In [15]:
from prisma import Translator

translator = Translator(llm, verbose=False)
tr = translator.translate(EXAMPLE_QUERY)

print("=" * 70)
print("TRANSLATOR HIGH-LEVEL RESULT")
print("=" * 70)
print(f"  Success          : {tr.success}")
print(f"  Query type       : {tr.intent_dict.get('query_type')}")
print(f"  Resources        : {tr.intent_dict.get('required_resources')}")
print(f"  Total steps      : {tr.total_steps}")
print(f"  Uses SERVICE     : {tr.uses_service}")
print(f"  Has aggregation  : {tr.has_aggregation}")
print(f"  Time             : {tr.processing_time_ms:.1f} ms")
if tr.intent_warnings:
    print(f"  Intent warnings  : {tr.intent_warnings}")
if tr.assembly_warnings:
    print(f"  Assembly warnings: {tr.assembly_warnings}")
print(f"\nGenerated SPARQL:")
print(tr.sparql_query)

{'query_type': 'complit_semantic', 'required_resources': ['complit', 'liita_lemma_bank'], 'lemma': 'colore', 'lemma_b': None, 'lemmas': None, 'semantic_relation': 'hyponym', 'filters': [], 'aggregation': None, 'retrieve_definitions': True, 'retrieve_examples': False, 'include_italian_written_rep': True, 'complexity_score': 3, 'user_query': "Find the hyponyms of 'colore'", 'reasoning': "Semantic relation query: hyponyms of 'colore' retrieved from CompL-it (via SERVICE clause), then bridged to LiITA lemmas."}
TRANSLATOR HIGH-LEVEL RESULT
  Success          : True
  Query type       : complit_semantic
  Resources        : ['complit', 'liita_lemma_bank']
  Total steps      : 3
  Uses SERVICE     : True
  Has aggregation  : False
  Time             : 1.0 ms

Generated SPARQL:
PREFIX lexinfo: <http://www.lexinfo.net/ontology/3.0/lexinfo#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/cor

In [16]:
# Confirm that the step-by-step result matches the high-level result
manual  = assembly_result.sparql_query.strip()
auto    = tr.sparql_query.strip()

if manual == auto:
    print("Step-by-step result matches the high-level Translator output.")
    print(f"  ({len(manual)} characters, identical)")
else:
    print("Results differ (can happen when a real LLM gives a slightly different response).")
    from difflib import unified_diff
    diff = list(unified_diff(manual.splitlines(), auto.splitlines(),
                             fromfile="step-by-step", tofile="Translator", lineterm=""))
    if diff:
        print("\nDiff:")
        print("\n".join(diff))
    else:
        print("(diff is empty — strings are semantically identical)")

Step-by-step result matches the high-level Translator output.
  (864 characters, identical)


---
## 5 — More Examples

The table below shows how different query types produce different execution plans. We run each query through the full pipeline and print the plan and a condensed version of the resulting SPARQL.

In [17]:
# Pre-computed mock responses for 3 additional query types.
# Replace MOCK_RESPONSES with your own LLM client to run live.

MOCK_RESPONSES = {
    "What is the polarity of 'amore'?": json.dumps({
        "query_type": "sentix_polarity",
        "required_resources": ["liita_lemma_bank", "sentix"],
        "lemma": "amore",
        "lemma_b": None,
        "lemmas": None,
        "semantic_relation": None,
        "filters": [],
        "aggregation": None,
        "retrieve_definitions": False,
        "retrieve_examples": False,
        "include_italian_written_rep": True,
        "complexity_score": 2,
        "user_query": "What is the polarity of 'amore'?",
        "reasoning": "Sentiment/polarity lookup for a specific lemma using Sentix"
    }),
    "What emotion is associated with 'felicità'?": json.dumps({
        "query_type": "elita_emotion",
        "required_resources": ["liita_lemma_bank", "elita"],
        "lemma": "felicità",
        "lemma_b": None,
        "lemmas": None,
        "semantic_relation": None,
        "filters": [],
        "aggregation": None,
        "retrieve_definitions": False,
        "retrieve_examples": False,
        "include_italian_written_rep": True,
        "complexity_score": 2,
        "user_query": "What emotion is associated with 'felicità'?",
        "reasoning": "Emotion lookup for a specific lemma using ELIta"
    }),
    "How many nouns are in LiITA?": json.dumps({
        "query_type": "basic_lemma_lookup",
        "required_resources": ["liita_lemma_bank"],
        "lemma": None,
        "lemma_b": None,
        "lemmas": None,
        "semantic_relation": None,
        "filters": [{"property": "pos", "value": "noun"}],
        "aggregation": {
            "type": "count",
            "aggregate_var": None,
            "group_by_vars": [],
            "separator": None,
            "order_by": None
        },
        "retrieve_definitions": False,
        "retrieve_examples": False,
        "include_italian_written_rep": False,
        "complexity_score": 1,
        "user_query": "How many nouns are in LiITA?",
        "reasoning": "Simple count aggregation on LiITA lemmas filtered by POS"
    }),
}


class MultiQueryMockLLM:
    """Mock LLM that dispatches to per-query pre-computed responses."""
    provider_name = "Mock"
    model         = "mock-1.0"

    def complete(self, prompt, system=None, temperature=None, max_tokens=None, **kwargs):
        for query, response in MOCK_RESPONSES.items():
            if query in prompt:
                return response
        return json.dumps({   # fallback
            "query_type": "basic_lemma_lookup",
            "required_resources": ["liita_lemma_bank"],
            "lemma": None, "lemma_b": None, "lemmas": None,
            "semantic_relation": None, "filters": [], "aggregation": None,
            "retrieve_definitions": False, "retrieve_examples": False,
            "include_italian_written_rep": True,
            "complexity_score": 1,
            "user_query": "unknown",
            "reasoning": "fallback"
        })


example_translator = Translator(MultiQueryMockLLM() if USE_MOCK_LLM else llm)

for query in MOCK_RESPONSES:
    result = example_translator.translate(query)

    # Reconstruct the plan from the translation result (it is stored inside)
    plan_ex = result.execution_plan

    print("=" * 70)
    print(f"Query : {query!r}")
    print(f"  query_type  : {result.intent_dict.get('query_type')}")
    print(f"  resources   : {result.intent_dict.get('required_resources')}")
    print(f"  reasoning   : {result.intent_dict.get('reasoning')}")
    print(f"  Plan ({len(plan_ex.steps)} step(s)):")
    for step in plan_ex.steps:
        print(f"    {step.step_number}. {step.tool_name}  — {step.description}")
    print(f"\n  SPARQL (condensed):")
    # Show first 12 non-empty lines
    sparql_lines = [l for l in result.sparql_query.splitlines() if l.strip()]
    for line in sparql_lines[:12]:
        print(f"    {line}")
    if len(sparql_lines) > 12:
        print(f"    ... ({len(sparql_lines) - 12} more lines)")
    print()

{'query_type': 'sentix_polarity', 'required_resources': ['liita_lemma_bank', 'sentix'], 'lemma': 'amore', 'lemma_b': None, 'lemmas': None, 'semantic_relation': None, 'filters': [], 'aggregation': None, 'retrieve_definitions': False, 'retrieve_examples': False, 'include_italian_written_rep': True, 'complexity_score': 2, 'user_query': "What is the polarity of 'amore'?", 'reasoning': 'Sentiment/polarity lookup for a specific lemma using Sentix'}
Query : "What is the polarity of 'amore'?"
  query_type  : sentix_polarity
  resources   : ['liita_lemma_bank', 'sentix']
  reasoning   : Sentiment/polarity lookup for a specific lemma using Sentix
  Plan (2 step(s)):
    1. liita_basic_query  — Query LiITA for lemma amore
    2. sentix_linking  — Link to Sentix affective lexicon

  SPARQL (condensed):
    PREFIX lila: <http://lila-erc.eu/ontologies/lila/>
    PREFIX marl: <http://www.gsi.upm.es/ontologies/marl/ns#>
    PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
    SELECT ?lemma ?wr ?s

---
## 6 — (Optional) Execute Against the LiITA Endpoint

The SPARQL queries generated above can be executed directly against the LiITA endpoint at `https://liita.it/sparql`. The cell below sends the query from Section 3 and prints the first few results.

> **Note**: this cell requires a network connection to `liita.it`.

In [18]:
EXECUTE_QUERY = True  # set True to run against the live endpoint

if EXECUTE_QUERY:
    import urllib.request, urllib.parse

    ENDPOINT = "https://liita.it/sparql"
    sparql   = assembly_result.sparql_query

    print(f"Executing against {ENDPOINT} ...\n")

    params  = urllib.parse.urlencode({"query": sparql, "format": "application/sparql-results+json"})
    request = urllib.request.Request(
        ENDPOINT,
        data=params.encode(),
        headers={"Accept": "application/sparql-results+json",
                 "Content-Type": "application/x-www-form-urlencoded"}
    )

    with urllib.request.urlopen(request, timeout=30) as resp:
        data = json.loads(resp.read())

    bindings = data.get("results", {}).get("bindings", [])
    print(f"Results: {len(bindings)} row(s) returned\n")

    # Print first 10 rows
    for i, row in enumerate(bindings[:10]):
        row_str = {k: v.get("value", "") for k, v in row.items()}
        print(f"  Row {i+1:2d}: {row_str}")

    if len(bindings) > 10:
        print(f"  ... ({len(bindings) - 10} more rows)")
else:
    print("Execution skipped (set EXECUTE_QUERY = True to run against the live endpoint).")

Executing against https://liita.it/sparql ...



Results: 282 row(s) returned

  Row  1: {'relatedWord': 'http://lexica/mylexicon#MUSlavandaNOUN', 'relatedSense': 'http://lexica/mylexicon#USem4060lavanda', 'definition': 'colore', 'liitaLemma': 'http://liita.it/data/id/lemma/1005421', 'italianWR': 'lavanda'}
  Row  2: {'relatedWord': 'http://lexica/mylexicon#MUSlavandaNOUN', 'relatedSense': 'http://lexica/mylexicon#USem4060lavanda', 'definition': 'una camicetta lavanda', 'liitaLemma': 'http://liita.it/data/id/lemma/1005421', 'italianWR': 'lavanda'}
  Row  3: {'relatedWord': 'http://lexica/mylexicon#MUSmagentaNOUN', 'relatedSense': 'http://lexica/mylexicon#USemTH35985magenta', 'definition': 'Colore cremisi molto intenso', 'liitaLemma': 'http://liita.it/data/id/lemma/1007746', 'italianWR': 'magenta'}
  Row  4: {'relatedWord': 'http://lexica/mylexicon#MUSmieleNOUN', 'relatedSense': 'http://lexica/mylexicon#USem76886miele', 'definition': 'colore giallo ambrato', 'liitaLemma': 'http://liita.it/data/id/lemma/1010947', 'italianWR': 'mele'}
 

---
## Summary

| Stage | Component | Input | Output | LLM? |
|---|---|---|---|---|
| 1 | `IntentAnalyzer` | Natural language query | Intent dict (JSON) | **Yes** |
| 1b | `IntentConverter` | Intent dict | Typed `Intent` object | No |
| 2 | `PatternOrchestrator` | `Intent` object | `ExecutionPlan` (list of `PatternStep`) | No |
| 3 | `PatternAssembler` | `ExecutionPlan` | `AssemblyResult` (SPARQL + fragments) | No |

The `Translator` class chains all of the above into a single `translate(query)` call.

**The key design principle**: the LLM is responsible only for intent classification (a narrow, well-defined task). All SPARQL generation is handled by hand-crafted, deterministic pattern tools — making the system transparent, debuggable, and free from hallucinated SPARQL syntax.